# Sin balancear

In [3]:
# -*- coding: utf-8 -*-
import os, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix
from types import SimpleNamespace

# =====================================
# CONFIGURACIÓN
# =====================================

# ⚠️ Solo las dimensiones solicitadas
RDIMS = range(1,17)

# Rutas Base
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
RESULTS_BASE = r"C:\Users\56946\TuckER\results"

# Ruta del TuckER 2020-2 (o el global)
DATA_DIR  = r"C:\Users\56946\TuckER\data\dataset_20192_fundamentales" 

# Carpeta de Predictores
PREDICTOR_BASE = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start\predictores_sem2_2019"

# Prefijos de archivos
RUN_PREFIX   = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_patience300"
PREDICTOR_NAME = "best_predictor_model_rdim{rdim}_normalizado_2019.pt"

# --- ESCENARIO TEMPORAL ---
CSV_HISTORIA = os.path.join(BASE_PATH, "df_20201.csv")  # Input (Solo Semestre 1)
CSV_TARGET   = os.path.join(BASE_PATH, "df_20202.csv")  # Target (Semestre 2)

# Cursos
CURSOS_PRIMER  = ['MA1101','MA1001','FI1000','BT1211'] 
CURSOS_SEGUNDO = ['MA1002','MA1102','FI1100','CC1002']

# Evaluamos predicción en todo lo que tomen (1ro o 2do) en el segundo semestre
CURSOS_EVAL    = CURSOS_PRIMER + CURSOS_SEGUNDO 

DEVICE = "cpu"
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# =========================
# FUNCIONES
# =========================
def get_vocab_manual(data_dir):
    entities, relations = set(), set()
    for part in ['train.txt', 'valid.txt', 'test.txt']:
        path = os.path.join(data_dir, part)
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip(): continue
                    h, r, t = line.strip().split()
                    entities.add(h.strip().upper()); entities.add(t.strip().upper()); relations.add(r.strip())
    entities = sorted(list(entities))
    relations = sorted(list(relations))
    relations_full = sorted(list(set(relations + [r + "_reverse" for r in relations])))
    return SimpleNamespace(entities=entities, entity_idxs={e: i for i, e in enumerate(entities)}, relation_idxs={r: i for i, r in enumerate(relations_full)})

def load_tucker_weights(path, device="cpu"):
    ckpt = torch.load(path, map_location=device)
    sd = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
    return sd["E.weight"].to(device), sd["R.weight"].to(device), sd["W"].to(device)

def contract_M(W, r_vec): return torch.tensordot(W, r_vec, dims=([0],[0]))

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), 
            nn.ReLU(), nn.Dropout(0.3), 
            nn.Linear(64, 128), 
            nn.ReLU(), nn.Dropout(0.3), 
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def limpiar_nota(nota_str, estado):
    if isinstance(estado, str) and "Reprobado" in estado: return 1.0
    try: return float(str(nota_str).replace(",", ".")) if not pd.isna(nota_str) else 0.0
    except: return 0.0

def precalcular_vectores_sem1(alumnos_ids, df_historia, idx_primer):
    vectores_np = {aid: np.zeros(4, dtype=np.float32) for aid in alumnos_ids}
    df_f = df_historia[df_historia["ID"].isin(alumnos_ids)]
    
    for _, row in df_f.iterrows():
        if row["CURSO"] in idx_primer:
            vectores_np[row["ID"]][idx_primer[row["CURSO"]]] = limpiar_nota(row["NOTA"], row["ESTADO_CURSO"])
            
    return {aid: torch.tensor(vec/7.0).view(1, -1) for aid, vec in vectores_np.items()}

# =========================
# EVALUACIÓN
# =========================
def evaluar_modelo(tag, tucker_ckpt, predictor_ckpt, vocab):
    print(f"\n=============== Evaluando modelo: {tag} (BALANCEADO) ===============")
    
    # Cargar modelos
    try:
        E, R, W = load_tucker_weights(tucker_ckpt, device=DEVICE)
        d_e = E.shape[1]
        
        # Input size 4 (Semestre 1)
        predictor = EmbeddingPredictor(input_size=4, output_size=d_e).to(DEVICE)
        predictor.load_state_dict(torch.load(predictor_ckpt, map_location=DEVICE))
        predictor.eval()
        
        idx_apr = vocab.relation_idxs["aprueba"]
        idx_repr = vocab.relation_idxs["reprueba"]
    except Exception as e:
        print(f"❌ Error cargando modelos: {e}")
        return None

    # Cargar Datos
    df_hist = pd.read_csv(CSV_HISTORIA, sep=';')
    df_tgt  = pd.read_csv(CSV_TARGET, sep=';')
    
    for df in [df_hist, df_tgt]:
        df['ID'] = df['ID'].astype(str).str.strip().str.upper()
        df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

    # Cohorte 2020 (4 cursos en 2020-1)
    df_fund = df_hist[df_hist["CURSO"].isin(CURSOS_PRIMER)]
    conteo = df_fund.groupby("ID")["CURSO"].nunique()
    alumnos_validos = set(conteo[conteo == 4].index)
    
    # Precalcular
    idx_primer = {c: i for i, c in enumerate(CURSOS_PRIMER)}
    vectores_x = precalcular_vectores_sem1(alumnos_validos, df_hist, idx_primer)
    
    # Filtrar Target
    df_eval = df_tgt[(df_tgt['ID'].isin(alumnos_validos)) & (df_tgt['CURSO'].isin(CURSOS_EVAL))].copy()
    df_eval = df_eval[df_eval['CURSO'].isin(vocab.entity_idxs.keys())]
    
    def get_real_label(row):
        estado = str(row['ESTADO_CURSO'])
        if "Aprobado" in estado: return 1 
        if "Reprobado" in estado: return 0 
        try: return 1 if float(str(row['NOTA']).replace(",", ".")) >= 4.0 else 0
        except: return 0
        
    df_eval['y_true'] = df_eval.apply(get_real_label, axis=1)

    # Inferencia
    y_true_list = []
    y_pred_list = []

    with torch.no_grad():
        M_apr = contract_M(W, R[idx_apr])
        M_repr = contract_M(W, R[idx_repr])
        
        for _, r in df_eval.iterrows():
            aid, curso = r['ID'], r['CURSO']
            
            if aid not in vectores_x: continue
            
            x_vec = vectores_x[aid].to(DEVICE)
            e_h = predictor(x_vec).squeeze(0)
            e_t = E[vocab.entity_idxs[curso]]
            
            s_apr = torch.sigmoid((e_h.view(1,d_e) @ M_apr @ e_t.view(d_e,1)).squeeze()).item()
            s_repr = torch.sigmoid((e_h.view(1,d_e) @ M_repr @ e_t.view(d_e,1)).squeeze()).item()
            
            # Criterio estándar: >=
            pred_class = 1 if s_apr >= s_repr else 0
            
            y_true_list.append(r['y_true'])
            y_pred_list.append(pred_class)

    # Métricas
    cm = confusion_matrix(y_true_list, y_pred_list)
    tn, fp, fn, tp = cm.ravel()
    
    # --- CÁLCULO PARA EL REPORTE DETALLADO ---
    # 0 = Reprueba, 1 = Aprueba
    # TN: Real 0, Pred 0 -> Correcto Reprobado
    # FP: Real 0, Pred 1 -> Real Reprueba, Predijo Aprueba (Peligroso)
    # FN: Real 1, Pred 0 -> Real Aprueba, Predijo Reprueba (Falsa Alarma)
    # TP: Real 1, Pred 1 -> Correcto Aprobado
    
    total_reales_reprobados = tn + fp
    total_predichos_reprobados = tn + fn
    detectados_reprobados = tn
    perdidos_peligrosos = fp
    falsas_alarmas = fn
    
    # Evitar división por cero
    recall_reprobados = tn / total_reales_reprobados if total_reales_reprobados > 0 else 0
    precision_reprobados = tn / total_predichos_reprobados if total_predichos_reprobados > 0 else 0
    accuracy_global = (tn + tp) / len(y_true_list)

    print("\n" + "="*60)
    print(f" 📊 RESULTADOS CRITERIO ESTÁNDAR (RDIM {tag})")
    print("="*60)
    print(f"Total Evaluaciones              : {len(y_true_list)}")
    print(f"Accuracy Global (Hit@1)         : {accuracy_global:.4f}")
    print("-" * 60)
    print(f"Total REALES Reprobados         : {total_reales_reprobados}")
    print(f"Total PREDICHOS Reprobados      : {total_predichos_reprobados}")
    print("-" * 60)
    print(f"✅ RECALL (Sensibilidad) Reprueba: {recall_reprobados:.4f}")
    print(f"   (Detectamos {detectados_reprobados} de {total_reales_reprobados} reprobados)")
    print("-" * 60)
    print(f"🎯 PRECISION Reprueba           : {precision_reprobados:.4f}")
    print(f"   (De los {total_predichos_reprobados} que dijimos que reprobaban, {detectados_reprobados} eran reales)")
    print("="*60)
    
    print("\nDesglose:")
    print(f"Correctos Reprobados : {detectados_reprobados}")
    print(f"Perdidos (Peligrosos): {perdidos_peligrosos} (Predijo Aprueba, era Reprueba)")
    print(f"Falsas Alarmas       : {falsas_alarmas} (Predijo Reprueba, era Aprueba)")
    
    return {
        "rdim": tag,
        "Accuracy": accuracy_global,
        "Recall_Rep": recall_reprobados,
        "Precision_Rep": precision_reprobados,
        "N_Rep_Reales": total_reales_reprobados,
        "N_Rep_Detectados": detectados_reprobados
    }

def main():
    print("--- Cargando Vocabulario ---")
    vocab = get_vocab_manual(DATA_DIR)
    res_list = []
    
    print(f"\n--- Iniciando Evaluación S1->S2 para RDIMS: {RDIMS} ---")
    
    for rdim in RDIMS:
        tag = f"rdim{rdim}"
        tucker_ckpt = os.path.join(RESULTS_BASE, RUN_PREFIX.format(rdim=rdim), "best_model.pt")
        predictor_ckpt = os.path.join(PREDICTOR_BASE, PREDICTOR_NAME.format(rdim=rdim))

        if not os.path.exists(tucker_ckpt):
            print(f"⏩ Saltando {tag} (Falta TuckER)")
            continue
        if not os.path.exists(predictor_ckpt):
            print(f"⏩ Saltando {tag} (Falta Predictor)")
            continue

        res = evaluar_modelo(tag, tucker_ckpt, predictor_ckpt, vocab)
        if res:
            res_list.append(res)

    if res_list:
        df_sum = pd.DataFrame(res_list)
        # Guardar
        out_path = os.path.join(PREDICTOR_BASE, "resumen_s1_s2_rdims_seleccionadas_balanceado.csv")
        df_sum.to_csv(out_path, index=False)
        print(f"\n💾 Guardado en: {out_path}")
    else:
        print("⚠️ No se generaron resultados.")

if __name__ == "__main__":
    main()

--- Cargando Vocabulario ---

--- Iniciando Evaluación S1->S2 para RDIMS: range(1, 17) ---

=============== Evaluando modelo: rdim1 (BALANCEADO) ===============

 📊 RESULTADOS CRITERIO ESTÁNDAR (RDIM rdim1)
Total Evaluaciones              : 3021
Accuracy Global (Hit@1)         : 0.7143
------------------------------------------------------------
Total REALES Reprobados         : 184
Total PREDICHOS Reprobados      : 839
------------------------------------------------------------
✅ RECALL (Sensibilidad) Reprueba: 0.4348
   (Detectamos 80 de 184 reprobados)
------------------------------------------------------------
🎯 PRECISION Reprueba           : 0.0954
   (De los 839 que dijimos que reprobaban, 80 eran reales)

Desglose:
Correctos Reprobados : 80
Perdidos (Peligrosos): 104 (Predijo Aprueba, era Reprueba)
Falsas Alarmas       : 759 (Predijo Reprueba, era Aprueba)

=============== Evaluando modelo: rdim2 (BALANCEADO) ===============

 📊 RESULTADOS CRITERIO ESTÁNDAR (RDIM rdim2)
Total 


 📊 RESULTADOS CRITERIO ESTÁNDAR (RDIM rdim10)
Total Evaluaciones              : 3021
Accuracy Global (Hit@1)         : 0.9454
------------------------------------------------------------
Total REALES Reprobados         : 184
Total PREDICHOS Reprobados      : 97
------------------------------------------------------------
✅ RECALL (Sensibilidad) Reprueba: 0.3152
   (Detectamos 58 de 184 reprobados)
------------------------------------------------------------
🎯 PRECISION Reprueba           : 0.5979
   (De los 97 que dijimos que reprobaban, 58 eran reales)

Desglose:
Correctos Reprobados : 58
Perdidos (Peligrosos): 126 (Predijo Aprueba, era Reprueba)
Falsas Alarmas       : 39 (Predijo Reprueba, era Aprueba)

=============== Evaluando modelo: rdim11 (BALANCEADO) ===============

 📊 RESULTADOS CRITERIO ESTÁNDAR (RDIM rdim11)
Total Evaluaciones              : 3021
Accuracy Global (Hit@1)         : 0.9454
------------------------------------------------------------
Total REALES Reprobados   

# Balanceados

In [2]:
# -*- coding: utf-8 -*-
import os, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix
from types import SimpleNamespace

# =====================================
# CONFIGURACIÓN
# =====================================

# Dimensiones a evaluar
RDIMS = range(1,17)

# Rutas Base
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
RESULTS_BASE = r"C:\Users\56946\TuckER\results"

# ⚠️ DATASET DE ENTRENAMIENTO (Para cargar vocabulario y TuckER)
# Usamos el del 2019-2 que fue el target del entrenamiento
DATA_DIR  = r"C:\Users\56946\TuckER\data\dataset_20192_fundamentales" 

# ⚠️ CARPETA DE PREDICTORES (S1 -> S2)
PREDICTOR_BASE = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start\predictores_sem1_balanceados"

# ⚠️ NOMBRES DE ARCHIVOS (Versión Balanceada S1->S2)
RUN_PREFIX     = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019patience400_balanceado"
PREDICTOR_NAME = "best_predictor_dim4_rdim{rdim}_2019_balanceado.pt"

# --- ESCENARIO DE EVALUACIÓN (2020) ---
CSV_HISTORIA = os.path.join(BASE_PATH, "df_20201.csv")  # Input (Solo Semestre 1)
CSV_TARGET   = os.path.join(BASE_PATH, "df_20202.csv")  # Target (Semestre 2)

# Cursos
CURSOS_PRIMER  = ['MA1101','MA1001','FI1000','BT1211'] 
CURSOS_SEGUNDO = ['MA1002','MA1102','FI1100','CC1002']

# Evaluamos predicción en todo lo que tomen (1ro o 2do) en el segundo semestre
CURSOS_EVAL    = CURSOS_PRIMER + CURSOS_SEGUNDO 

DEVICE = "cpu"
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# =========================
# FUNCIONES
# =========================
def get_vocab_manual(data_dir):
    entities, relations = set(), set()
    # Intentamos leer train.txt (si lo renombraste) o train_balanceado.txt
    nombres_posibles = ['train.txt', 'train_balanceado.txt', 'valid.txt', 'test.txt']
    
    for part in nombres_posibles:
        path = os.path.join(data_dir, part)
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip(): continue
                    h, r, t = line.strip().split()
                    entities.add(h.strip().upper()); entities.add(t.strip().upper()); relations.add(r.strip())
    
    entities = sorted(list(entities))
    relations = sorted(list(relations))
    relations_full = sorted(list(set(relations + [r + "_reverse" for r in relations])))
    return SimpleNamespace(entities=entities, entity_idxs={e: i for i, e in enumerate(entities)}, relation_idxs={r: i for i, r in enumerate(relations_full)})

def load_tucker_weights(path, device="cpu"):
    ckpt = torch.load(path, map_location=device)
    sd = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
    return sd["E.weight"].to(device), sd["R.weight"].to(device), sd["W"].to(device)

def contract_M(W, r_vec): return torch.tensordot(W, r_vec, dims=([0],[0]))

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), 
            nn.ReLU(), nn.Dropout(0.3), 
            nn.Linear(64, 128), 
            nn.ReLU(), nn.Dropout(0.3), 
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def limpiar_nota(nota_str, estado):
    if isinstance(estado, str) and "Reprobado" in estado: return 1.0
    try: return float(str(nota_str).replace(",", ".")) if not pd.isna(nota_str) else 0.0
    except: return 0.0

def precalcular_vectores_sem1(alumnos_ids, df_historia, idx_primer):
    # Vector tamaño 4
    vectores_np = {aid: np.zeros(4, dtype=np.float32) for aid in alumnos_ids}
    df_f = df_historia[df_historia["ID"].isin(alumnos_ids)]
    
    for _, row in df_f.iterrows():
        if row["CURSO"] in idx_primer:
            vectores_np[row["ID"]][idx_primer[row["CURSO"]]] = limpiar_nota(row["NOTA"], row["ESTADO_CURSO"])
            
    return {aid: torch.tensor(vec/7.0).view(1, -1) for aid, vec in vectores_np.items()}

# =========================
# EVALUACIÓN
# =========================
def evaluar_modelo(tag, tucker_ckpt, predictor_ckpt, vocab):
    print(f"\n=============== Evaluando modelo: {tag} (BALANCEADO S1->S2) ===============")
    
    try:
        E, R, W = load_tucker_weights(tucker_ckpt, device=DEVICE)
        d_e = E.shape[1]
        
        # ⚠️ Input size 4 (Solo Semestre 1)
        predictor = EmbeddingPredictor(input_size=4, output_size=d_e).to(DEVICE)
        predictor.load_state_dict(torch.load(predictor_ckpt, map_location=DEVICE))
        predictor.eval()
        
        idx_apr = vocab.relation_idxs["aprueba"]
        idx_repr = vocab.relation_idxs["reprueba"]
    except Exception as e:
        print(f"❌ Error cargando modelos: {e}")
        return None

    # Cargar Datos 2020
    df_hist = pd.read_csv(CSV_HISTORIA, sep=';')
    df_tgt  = pd.read_csv(CSV_TARGET, sep=';')
    
    for df in [df_hist, df_tgt]:
        df['ID'] = df['ID'].astype(str).str.strip().str.upper()
        df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

    # Cohorte 2020 (Deben tener los 4 del primer semestre)
    df_fund = df_hist[df_hist["CURSO"].isin(CURSOS_PRIMER)]
    conteo = df_fund.groupby("ID")["CURSO"].nunique()
    alumnos_validos = set(conteo[conteo == 4].index)
    
    # Precalcular
    idx_primer = {c: i for i, c in enumerate(CURSOS_PRIMER)}
    vectores_x = precalcular_vectores_sem1(alumnos_validos, df_hist, idx_primer)
    
    # Filtrar Target (2020-2)
    df_eval = df_tgt[(df_tgt['ID'].isin(alumnos_validos)) & (df_tgt['CURSO'].isin(CURSOS_EVAL))].copy()
    df_eval = df_eval[df_eval['CURSO'].isin(vocab.entity_idxs.keys())]
    
    def get_real_label(row):
        estado = str(row['ESTADO_CURSO'])
        if "Aprobado" in estado: return 1 
        if "Reprobado" in estado: return 0 
        try: return 1 if float(str(row['NOTA']).replace(",", ".")) >= 4.0 else 0
        except: return 0
        
    df_eval['y_true'] = df_eval.apply(get_real_label, axis=1)

    # Inferencia
    y_true_list = []
    y_pred_list = []

    with torch.no_grad():
        M_apr = contract_M(W, R[idx_apr])
        M_repr = contract_M(W, R[idx_repr])
        
        for _, r in df_eval.iterrows():
            aid, curso = r['ID'], r['CURSO']
            
            if aid not in vectores_x: continue
            
            x_vec = vectores_x[aid].to(DEVICE)
            e_h = predictor(x_vec).squeeze(0)
            e_t = E[vocab.entity_idxs[curso]]
            
            s_apr = torch.sigmoid((e_h.view(1,d_e) @ M_apr @ e_t.view(d_e,1)).squeeze()).item()
            s_repr = torch.sigmoid((e_h.view(1,d_e) @ M_repr @ e_t.view(d_e,1)).squeeze()).item()
            
            # Criterio estándar: >=
            pred_class = 1 if s_apr >= s_repr else 0
            
            y_true_list.append(r['y_true'])
            y_pred_list.append(pred_class)

    # Métricas
    cm = confusion_matrix(y_true_list, y_pred_list)
    tn, fp, fn, tp = cm.ravel()
    
    total_reales_reprobados = tn + fp
    total_predichos_reprobados = tn + fn
    detectados_reprobados = tn
    perdidos_peligrosos = fp
    falsas_alarmas = fn
    
    recall_reprobados = tn / total_reales_reprobados if total_reales_reprobados > 0 else 0
    precision_reprobados = tn / (tn + fn) if (tn + fn) > 0 else 0
    accuracy_global = (tn + tp) / len(y_true_list)

    print("\n" + "="*60)
    print(f" 📊 RESULTADOS CRITERIO ESTÁNDAR ({tag})")
    print("="*60)
    print(f"Total Evaluaciones              : {len(y_true_list)}")
    print(f"Accuracy Global (Hit@1)         : {accuracy_global:.4f}")
    print("-" * 60)
    print(f"Total REALES Reprobados         : {total_reales_reprobados}")
    print(f"Total PREDICHOS Reprobados      : {total_predichos_reprobados}")
    print("-" * 60)
    print(f"✅ RECALL (Sensibilidad) Reprueba: {recall_reprobados:.4f}")
    print(f"   (Detectamos {detectados_reprobados} de {total_reales_reprobados} reprobados)")
    print("-" * 60)
    print(f"🎯 PRECISION Reprueba           : {precision_reprobados:.4f}")
    print(f"   (De los {total_predichos_reprobados} que dijimos que reprobaban, {detectados_reprobados} eran reales)")
    print("="*60)
    
    print("\nDesglose:")
    print(f"Correctos Reprobados : {detectados_reprobados}")
    print(f"Perdidos (Peligrosos): {perdidos_peligrosos} (Predijo Aprueba, era Reprueba)")
    print(f"Falsas Alarmas       : {falsas_alarmas} (Predijo Reprueba, era Aprueba)")
    
    return {
        "rdim": tag,
        "Accuracy": accuracy_global,
        "Recall_Rep": recall_reprobados,
        "Precision_Rep": precision_reprobados
    }

def main():
    print("--- Cargando Vocabulario ---")
    vocab = get_vocab_manual(DATA_DIR)
    res_list = []
    
    print(f"\n--- Iniciando Evaluación S1->S2 para RDIMS: {RDIMS} ---")
    
    for rdim in RDIMS:
        tag = f"rdim{rdim}"
        tucker_ckpt = os.path.join(RESULTS_BASE, RUN_PREFIX.format(rdim=rdim), "best_model.pt")
        predictor_ckpt = os.path.join(PREDICTOR_BASE, PREDICTOR_NAME.format(rdim=rdim))

        if not os.path.exists(tucker_ckpt):
            print(f"⏩ Saltando {tag} (Falta TuckER)")
            continue
        if not os.path.exists(predictor_ckpt):
            print(f"⏩ Saltando {tag} (Falta Predictor)")
            continue

        res = evaluar_modelo(tag, tucker_ckpt, predictor_ckpt, vocab)
        if res:
            res_list.append(res)

    if res_list:
        df_sum = pd.DataFrame(res_list)
        out_path = os.path.join(PREDICTOR_BASE, "resumen_s1_s2_balanceado_final.csv")
        df_sum.to_csv(out_path, index=False)
        print(f"\n💾 Guardado en: {out_path}")
    else:
        print("⚠️ No se generaron resultados.")

if __name__ == "__main__":
    main()

--- Cargando Vocabulario ---

--- Iniciando Evaluación S1->S2 para RDIMS: range(1, 17) ---

=============== Evaluando modelo: rdim1 (BALANCEADO S1->S2) ===============

 📊 RESULTADOS CRITERIO ESTÁNDAR (rdim1)
Total Evaluaciones              : 3021
Accuracy Global (Hit@1)         : 0.9454
------------------------------------------------------------
Total REALES Reprobados         : 184
Total PREDICHOS Reprobados      : 97
------------------------------------------------------------
✅ RECALL (Sensibilidad) Reprueba: 0.3152
   (Detectamos 58 de 184 reprobados)
------------------------------------------------------------
🎯 PRECISION Reprueba           : 0.5979
   (De los 97 que dijimos que reprobaban, 58 eran reales)

Desglose:
Correctos Reprobados : 58
Perdidos (Peligrosos): 126 (Predijo Aprueba, era Reprueba)
Falsas Alarmas       : 39 (Predijo Reprueba, era Aprueba)

=============== Evaluando modelo: rdim2 (BALANCEADO S1->S2) ===============

 📊 RESULTADOS CRITERIO ESTÁNDAR (rdim2)
Total


 📊 RESULTADOS CRITERIO ESTÁNDAR (rdim10)
Total Evaluaciones              : 3021
Accuracy Global (Hit@1)         : 0.9454
------------------------------------------------------------
Total REALES Reprobados         : 184
Total PREDICHOS Reprobados      : 97
------------------------------------------------------------
✅ RECALL (Sensibilidad) Reprueba: 0.3152
   (Detectamos 58 de 184 reprobados)
------------------------------------------------------------
🎯 PRECISION Reprueba           : 0.5979
   (De los 97 que dijimos que reprobaban, 58 eran reales)

Desglose:
Correctos Reprobados : 58
Perdidos (Peligrosos): 126 (Predijo Aprueba, era Reprueba)
Falsas Alarmas       : 39 (Predijo Reprueba, era Aprueba)

=============== Evaluando modelo: rdim11 (BALANCEADO S1->S2) ===============

 📊 RESULTADOS CRITERIO ESTÁNDAR (rdim11)
Total Evaluaciones              : 3021
Accuracy Global (Hit@1)         : 0.7143
------------------------------------------------------------
Total REALES Reprobados      